# Data exploration and filtering

### Setup

In [1]:
%load_ext autoreload
%autoreload 2

import pandas as pd

from ml_powerlifting_potential import config
from ml_powerlifting_potential.data import load_raw, prepare_dataset

### Raw dataset overview

In [2]:
# number of columns in the full file (header only)
all_columns = pd.read_csv(config.RAW_CSV, nrows=0).columns
print(len(all_columns), "columns")

41 columns


In [3]:
# load only the useful columns
raw_data = load_raw()
raw_data.shape

(3043013, 14)

In [4]:
# share of missing values per column
raw_data.isna().mean().sort_values(ascending=False)

Age                0.351736
Best3SquatKg       0.331635
Best3DeadliftKg    0.274998
Tested             0.264310
Best3BenchKg       0.117833
TotalKg            0.066520
BodyweightKg       0.012609
Equipment          0.000000
Name               0.000000
Sex                0.000000
Event              0.000000
Place              0.000000
Federation         0.000000
Date               0.000000
dtype: float64

In [5]:
raw_data["Event"].value_counts()

Event
SBD    2107555
B       675638
D       177048
BD       61477
S        16428
SB        2998
SD        1869
Name: count, dtype: int64

In [6]:
raw_data["Equipment"].value_counts()

Equipment
Raw           1366090
Single-ply    1340888
Wraps          202616
Multi-ply      124316
Unlimited        9048
Straps             55
Name: count, dtype: int64

### Filtering funnel

In [7]:
# free memory before running the full pipeline
del raw_data

# same filtering as the real pipeline in data.py
dataset, funnel_df = prepare_dataset()
funnel_df

,step,rows_left,rows_removed
0,Raw dataset,3043013,NaN
1,SBD only,2107555,935458.0
2,Raw only,771491,1336064.0
3,Not disqualified (DQ),742893,28598.0
4,No doping disqualification (DD),742425,468.0
5,Squat >= 20 kg,740242,2183.0
6,Bench >= 20 kg,739795,447.0
7,Deadlift >= 20 kg,739746,49.0
8,Sex M or F,739692,54.0
9,Bodyweight >= 40 kg,737465,2227.0


### Final dataset

In [8]:
dataset.describe().round(1)

,Best3BenchKg,BodyweightKg,Age,Sex,Tested,Best3SquatKg,Best3DeadliftKg
count,227625.0,227625.0,227625.0,227625.0,227625.0,227625.0,227625.0
mean,107.7,84.0,28.1,0.7,0.8,165.7,193.5
std,42.1,20.9,10.3,0.5,0.4,56.9,57.9
min,20.0,40.0,16.0,0.0,0.0,20.0,20.0
25%,70.0,69.0,21.0,0.0,1.0,120.0,145.0
50%,110.0,81.6,25.5,1.0,1.0,165.0,197.5
75%,137.5,96.6,32.0,1.0,1.0,205.0,235.0
max,325.0,246.8,92.5,1.0,1.0,490.0,487.5


In [9]:
# must be 0 everywhere
dataset.isna().sum()

Best3BenchKg       0
BodyweightKg       0
Age                0
Sex                0
Tested             0
Best3SquatKg       0
Best3DeadliftKg    0
dtype: int64

In [10]:
# 1 = men, 0 = women
dataset["Sex"].value_counts()

Sex
1    155632
0     71993
Name: count, dtype: int64